<a href="https://colab.research.google.com/github/kayseenin/healthcare-ai-learning/blob/main/notebooks/01_file_search_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
## Setup

Before running this notebook:

1. Add your OpenAI API key as a Colab Secret named `OPEN_API_KEY`. Toggle **Notebook access** on. ([Get a key →](https://platform.openai.com/api-keys))
2. Download the [MassHealth Knee Arthroplasty PDF](https://www.mass.gov/doc/guidelines-for-medical-necessity-determination-for-knee-arthroplasty/download) and upload it to `/content/` in this Colab session (left sidebar → folder icon → drag-and-drop).

Then run cells in order.

SyntaxError: invalid character '→' (U+2192) (489508477.py, line 5)

In [ ]:
!pip install openai

In [ ]:
!pip install openai -q

from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY from env

# Create the vector store
vector_store = client.vector_stores.create(name="masshealth_knee_arthroplasty")
print(f"Vector store created: {vector_store.id}")

# Upload the PDF
with open(PDF_PATH, 'rb') as f:
    file_obj = client.files.create(file=f, purpose="assistants")
print(f"File uploaded:        {file_obj.id}")

# Attach file to vector store
attach = client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_obj.id,
)
print(f"Attached, status:     {attach.status}")

In [ ]:
import os
from google.colab import userdata

# Pull from YOUR secret name, store under the env var OpenAI expects
os.environ['OPENAI_API_KEY'] = userdata.get('OPEN_API_KEY')

assert os.environ.get('OPENAI_API_KEY'), "Bridge failed — check secret name and Notebook access toggle"
print("Bridge set: OPEN_API_KEY (secret) → OPENAI_API_KEY (env var)")

In [ ]:
import time

while True:
    f = client.vector_stores.files.retrieve(
        vector_store_id=vector_store.id,
        file_id=file_obj.id,
    )
    print(f"Status: {f.status}")
    if f.status == "completed":
        break
    if f.status == "failed":
        raise RuntimeError(f"Processing failed: {f.last_error}")
    time.sleep(2)

print(f"\nReady to query. Vector store: {vector_store.id}")

In [ ]:
query = "What are the three types of knee arthroplasty covered by these MassHealth guidelines?"

response = client.responses.create(
    input=query,
    model="gpt-4o-mini",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
    }],
)

print(f"Output items: {[item.type for item in response.output]}\n")

for item in response.output:
    if item.type == "message":
        print("ANSWER:")
        print(item.content[0].text)
        print()
        annotations = item.content[0].annotations
        if annotations:
            files = set(a.filename for a in annotations)
            print(f"Sources cited: {files}")
        else:
            print("No annotations returned — model may have answered from its own knowledge.")

In [ ]:
print(response.model_dump_json(indent=2))

In [13]:
test_questions = [
    "What is the BMI restriction for unicompartmental knee arthroplasty (UKA), and what are the exceptions for members above that BMI?",
    "Is robot-assisted total knee arthroplasty (Makoplasty) covered by MassHealth? Explain why or why not.",
    "What are the Kellgren-Lawrence grade requirements for total knee arthroplasty, and what does each grade describe?",
]

results = []

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {q}")
    print('='*70)

    response = client.responses.create(
        input=q,
        model="gpt-4o-mini",
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
        }],
        include=["output[*].file_search_call.search_results"],
    )

    answer, annotations, chunks = None, [], []
    for item in response.output:
        if item.type == "message":
            answer = item.content[0].text
            annotations = item.content[0].annotations or []
        elif item.type == "file_search_call":
            if hasattr(item, "search_results") and item.search_results:
                chunks = item.search_results

    print(f"\nANSWER:\n{answer}\n")
    print(f"CHUNKS RETRIEVED: {len(chunks)}")
    for j, c in enumerate(chunks[:3], 1):
        snippet = (c.content[0].text[:160].replace('\n', ' ')
                   if c.content else "(no content)")
        print(f"  [{j}] score={c.score:.3f}  {snippet}...")
    print(f"\nCITATIONS: {len(annotations)}")

    results.append({
        "question": q,
        "answer": answer,
        "num_chunks": len(chunks),
        "num_citations": len(annotations),
        "top_chunk_score": chunks[0].score if chunks else None,
    })

print(f"\n\n{'='*70}\nTested {len(results)} questions. Block 4 complete.")


Q1: What is the BMI restriction for unicompartmental knee arthroplasty (UKA), and what are the exceptions for members above that BMI?


NameError: name 'vector_store' is not defined

In [ ]:
test_questions = [
    "What CPT codes and documentation are required in a MassHealth prior authorization request for knee arthroplasty?",
    "For a member with a previous TKA, what criteria must be met before a revision arthroplasty is considered medically necessary?",
    "Can a patient who received an articular injection 2 months ago be approved for total knee arthroplasty?",
    "What is the typical cost or payment amount for knee arthroplasty under MassHealth?",
    "What is the effective date of these guidelines, and who approved them?",
]

results_block6 = []

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {q}")
    print('='*70)

    response = client.responses.create(
        input=q,
        model="gpt-4o-mini",
        temperature=0.2,
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
        }],
        include=["output[*].file_search_call.search_results"],
    )

    answer, annotations, chunks = None, [], []
    for item in response.output:
        if item.type == "message":
            answer = item.content[0].text
            annotations = item.content[0].annotations or []
        elif item.type == "file_search_call":
            if hasattr(item, "results") and item.results:
                chunks = item.results

    print(f"\nANSWER:\n{answer}\n")
    print(f"CHUNKS RETRIEVED: {len(chunks)}")
    for j, c in enumerate(chunks[:3], 1):
        snippet = c.text[:160].replace('\n', ' ') if c.text else "(no content)"
        print(f"  [{j}] score={c.score:.3f}  {snippet}...")
    print(f"\nCITATIONS: {len(annotations)}")

    results_block6.append({
        "question": q,
        "answer": answer,
        "num_chunks": len(chunks),
        "num_citations": len(annotations),
        "top_chunk_score": chunks[0].score if chunks else None,
    })

print(f"\n\n{'='*70}\nTested {len(results_block6)} questions. Block 6 complete.")